# GRPO in Practice — Group Relative Policy Optimization, Two Ways

**What this notebook is.** A hands-on walkthrough of *Group Relative Policy Optimization* (GRPO),
the RL algorithm behind DeepSeek-R1's reasoning training. It runs end-to-end on a free Colab T4.

**Why GRPO instead of PPO.** Classic RLHF (PPO) needs a separate **value/critic network** roughly
the size of the policy — expensive to train and to hold in memory. GRPO throws the critic away.
Instead, for each prompt it samples a **group** of `G` completions, scores them all with reward
functions, and uses the **group's own mean and standard deviation** as the baseline:

$$A_i = \frac{r_i - \text{mean}(r_1 \dots r_G)}{\text{std}(r_1 \dots r_G)}$$

That normalized score is the *advantage*. Completions that beat their siblings get pushed up;
ones that lose get pushed down. A KL penalty against the frozen reference model keeps the policy
from drifting into gibberish.

**The one thing to internalize:** in GRPO, *the reward function is the training data.* You are not
labelling outputs — you are writing code that grades them. Most of this notebook is reward design.

---

## Notebook layout

| Part | Model | Stack | Task | What it teaches |
|---|---|---|---|---|
| **Part 1** | SmolLM-135M-Instruct | TRL + PEFT | 8 hand-written Q&A pairs | The mechanics — reward funcs, `GRPOConfig`, LoRA, the training loop |
| **Part 2** | Qwen2.5-0.5B-Instruct | Unsloth + vLLM | GSM8K math word problems | The real thing — format rewards, verifiable correctness, fast rollouts |

Part 1 is a **toy**: similarity-to-reference rewards are a bad idea in production (the model learns
to parrot, not to reason). It exists so you can watch every moving part. Part 2 is the pattern you'd
actually ship: a **verifiable** reward (is the final number right?) plus format shaping.

**Runtime:** GPU required. Colab → Runtime → Change runtime type → T4 GPU.

---
# Part 1 — GRPO from first principles (TRL + LoRA)

## Lesson 1 — Environment

Four libraries do the work:

- **`trl`** — Hugging Face's RL library; supplies `GRPOTrainer` and `GRPOConfig`.
- **`peft`** — LoRA adapters, so we train ~1% of the parameters instead of all 135M.
- **`transformers` / `datasets`** — model loading and data plumbing.
- **`accelerate` / `bitsandbytes`** — device placement and quantized optimizers.

`torchao` is uninstalled because its quantization hooks conflict with the TRL/PEFT
combination on Colab and surface as opaque dtype errors mid-training.

> **Restart the runtime after this cell** if Colab prompts you to.

In [1]:
!pip install -q -U datasets transformers trl peft accelerate bitsandbytes

# torchao is PyTorch's quantization package. It clashes with the TRL + PEFT
# stack on Colab, so remove it before importing anything else.
!pip uninstall -y torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 19.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 115.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.6 MB/s eta 0:00:00
Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


## Lesson 2 — Imports

`SequenceMatcher` (stdlib `difflib`) is our stand-in reward model for Part 1 — it gives a cheap
string-similarity score without needing a second neural network on the GPU.

In [2]:
import re
import torch
from difflib import SequenceMatcher

from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, PeftModel
from trl import GRPOConfig, GRPOTrainer

## Lesson 3 — The dataset

Eight prompts, each with a `reference_answer` we'll grade against.

Note what GRPO does **not** need: this is *not* a supervised fine-tuning set. The model never sees
`reference_answer` as a target to imitate. It only ever sees the `prompt`, generates its own
attempts, and the reference is used **inside the reward function** to score them. That distinction
is the whole point of RL fine-tuning.

Eight examples is absurdly small — it's enough to watch reward curves move, not enough to actually
improve the model.

In [3]:
data = [
    {
        "prompt": "Explain reinforcement learning in simple terms.",
        "reference_answer": "Reinforcement learning is a type of machine learning where an agent learns by taking actions, receiving rewards or penalties, and improving over time.",
    },
    {
        "prompt": "Write a polite email for delayed delivery.",
        "reference_answer": "We apologize for the delay in your delivery. Your order is on the way, and we appreciate your patience and understanding.",
    },
    {
        "prompt": "What is a neural network?",
        "reference_answer": "A neural network is a machine learning model inspired by the human brain. It learns patterns from data using layers of connected nodes.",
    },
    {
        "prompt": "Explain overfitting in ML.",
        "reference_answer": "Overfitting happens when a model learns the training data too closely, including noise, and performs poorly on new unseen data.",
    },
    {
        "prompt": "Explain supervised learning in simple terms.",
        "reference_answer": "Supervised learning is a machine learning method where a model learns from labeled examples and then predicts outputs for new data.",
    },
    {
        "prompt": "What is gradient descent?",
        "reference_answer": "Gradient descent is an optimization method that helps a model reduce its error by slowly adjusting its parameters in the right direction.",
    },
    {
        "prompt": "Explain classification in machine learning.",
        "reference_answer": "Classification is a machine learning task where the model assigns input data to predefined categories or classes.",
    },
    {
        "prompt": "Write a simple apology message.",
        "reference_answer": "I am sorry for the inconvenience. Thank you for your patience and understanding.",
    },
]

### 3a — Prompt formatting

A fixed template is applied to every prompt. This matters more than it looks: the reward functions
implicitly assume a certain answer *shape*, and the template is what nudges the model toward it.
Keep this template identical between training and inference, or your trained behaviour won't show up.

In [ ]:
def format_prompt(example):
    """Wrap a raw question in the fixed instruction template used for training.

    The exact same string must be used at inference time (see `generate_answer`
    in Lesson 9) — GRPO conditions the learned behaviour on this prefix.

    Args:
        example: a dataset row containing a "prompt" key.

    Returns:
        dict: {"prompt": templated string} — `Dataset.map` merges this back in,
        overwriting the original "prompt" column.
    """
    return {
        "prompt": f"Question: {example['prompt']}\nAnswer in simple English:"
    }

In [5]:
dataset = Dataset.from_list(data)
dataset = dataset.map(format_prompt)

Map:   0%|          | 0/8 [00:00<?, ? examples/s]

In [6]:
dataset

Dataset({
    features: ['prompt', 'reference_answer'],
    num_rows: 8
})

## Lesson 4 — Model and tokenizer

**SmolLM-135M-Instruct** is deliberately tiny. GRPO generates `num_generations` completions for
*every* prompt at *every* step, so rollout cost dominates training cost — a small model keeps a
T4 demo under a few minutes.

Two tokenizer details that break things if you skip them:

- **`pad_token = eos_token`** — many causal LMs ship without a pad token; batching fails without one.
- **`padding_side = "left"`** — this is *mandatory* for generation. With right-padding, the model
  would continue from pad tokens instead of from the end of your prompt, and you'd get garbage.

In [7]:
model_id = "HuggingFaceTB/SmolLM-135M-Instruct"

In [8]:
tokenizer = AutoTokenizer.from_pretrained(model_id)

# Causal LMs often have no pad token — reuse EOS so batching works.
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# LEFT padding is required for generation: the model must continue from the
# real end of the prompt, not from a run of pad tokens.
tokenizer.padding_side = "left"

config.json:   0%|          | 0.00/723 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.59k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/565 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [13]:
MODEL = AutoModelForCausalLM.from_pretrained(
    model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

# Keep the model config in sync with the tokenizer, or generation warns/misbehaves.
MODEL.config.pad_token_id = tokenizer.pad_token_id

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/156 [00:00<?, ?B/s]

## Lesson 5 — LoRA

We freeze the base weights and train small low-rank adapters instead.

| Setting | Value | Why |
|---|---|---|
| `r=8` | rank of the update matrices | Higher = more capacity, more memory. 8–16 is standard. |
| `lora_alpha=16` | scaling factor | Effective scale is `alpha / r` = 2×. |
| `target_modules=["q_proj","v_proj"]` | which layers get adapters | Query + value projections give most of the benefit for the least memory. Add `k_proj`/`o_proj`/MLP projections for more capacity. |
| `lora_dropout=0.05` | regularization | Small dataset → keep some. |

There's a second reason LoRA suits GRPO specifically: the trainer needs a **frozen reference model**
for the KL penalty. With LoRA it can just disable the adapters to recover the reference — no second
copy of the weights in VRAM.

In [14]:
lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none",
)

---
## Lesson 6 — Reward functions (the core of GRPO)

Everything above was setup. **This is where the actual learning signal is defined.**

TRL calls each reward function with the batch and expects a list of floats — one score per
completion. The signature is:

```python
def my_reward(prompts, completions, **kwargs) -> list[float]
```

Any extra dataset column (like `reference_answer`) arrives as a keyword argument with the column's
name, holding a list aligned with `completions`. **Always accept `**kwargs`** — TRL passes several
things you may not use, and a strict signature raises `TypeError`.

We use three graders, and their scores are **summed**:

| Function | Range | Grades |
|---|---|---|
| `correctness_reward` | 0 → 5 | Does the answer match the reference in content? |
| `helpfulness_reward` | −2.5 → 1.5 | Is it complete and non-evasive? |
| `clarity_reward` | −0.5 → 1.0 | Is it readable, well-sized, not degenerate? |

Notice the implied weighting: correctness is worth ~3× the others, so the model prioritizes it.
Reward weighting is your main design lever — there's no separate loss to tune.

> **Reward hacking watch:** every one of these is gameable. `helpfulness_reward` pays for length,
> so the model may pad. `correctness_reward` pays for keyword overlap, so it may regurgitate the
> question's nouns. This is *normal* — you find the exploits by reading generations during training,
> then patch the reward. Part 2 shows the fix: reward something **verifiable**.

### 6a — Text normalization helper

In [ ]:
def normalize_text(text: str) -> str:
    """Lowercase, strip, and collapse all whitespace runs to single spaces.

    Used by every scorer so that "Hello   World " and "hello world" compare equal.
    Casing and spacing are noise we don't want to pay or penalize the model for.

    Args:
        text: any object; coerced to str first (completions aren't always strings).

    Returns:
        str: the normalized text.
    """
    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text

### 6b — Unwrapping completions

TRL hands you completions in one of two shapes depending on whether your dataset used plain-text
or conversational prompts:

- **plain text** → a `str`
- **chat format** → a list of message dicts, e.g. `[{"role": "assistant", "content": "..."}]`

This helper normalizes both so the scorers don't have to care.

In [ ]:
def get_completion_text(completion) -> str:
    """Extract the raw answer string from whatever shape TRL hands us.

    Handles three cases:
      * `str`  -> returned unchanged (plain-text prompt datasets).
      * `list` -> returns the "content" of the LAST message (chat-format datasets).
      * anything else -> coerced with `str()` so a scorer never crashes mid-run.

    Args:
        completion: a single generated completion from the trainer.

    Returns:
        str: the answer text.
    """
    if isinstance(completion, str):
        return completion

    if isinstance(completion, list):
        try:
            return completion[-1]["content"]
        except Exception:
            return str(completion)

    return str(completion)

### 6c — Similarity score

`SequenceMatcher.ratio()` finds the longest matching subsequences between two strings and returns
`2 * matches / total_length` — 1.0 for identical text, 0.0 for nothing in common. It's
order-sensitive, so it rewards phrasing that *tracks* the reference, not just shared vocabulary.

In [ ]:
def similarity_score(a: str, b: str) -> float:
    """Character-sequence similarity between two texts, in [0.0, 1.0].

    Order-sensitive: rewards answers whose wording follows the reference, not
    just answers that reuse its vocabulary. Both inputs are normalized first.

    Args:
        a: generated text.
        b: reference text.

    Returns:
        float: 1.0 = identical after normalization, 0.0 = nothing in common.
    """
    return SequenceMatcher(None, normalize_text(a), normalize_text(b)).ratio()

### 6d — Keyword overlap score

The order-insensitive counterpart. It asks: *what fraction of the reference's meaningful words did
the model produce?* Words shorter than 3 letters are dropped so "a", "is", "of" don't inflate the
score.

**Worked example**

```
a (generated) = "Machine learning uses data"
b (reference) = "Machine learning learns from data"

a_words = {machine, learning, uses, data}
b_words = {machine, learning, learns, from, data}
overlap = {machine, learning, data}  ->  3 / 5 = 0.6
```

Note the denominator is `b_words` (recall, not precision) — the model isn't penalized for saying
*extra* things, only for missing what the reference covered. `helpfulness_reward` handles rambling.

In [ ]:
def keyword_overlap_score(a: str, b: str) -> float:
    """Fraction of the reference's content words that appear in the generated text.

    Order-insensitive recall over words of >=3 letters (short function words are
    dropped so "a"/"is"/"of" can't inflate the score). Complements
    `similarity_score`, which is order-sensitive.

    Args:
        a: generated text.
        b: reference text.

    Returns:
        float: |a_words & b_words| / |b_words|, in [0.0, 1.0]. Returns 0.0 if the
        reference has no scorable words.
    """
    a_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(a)))
    b_words = set(re.findall(r"[a-zA-Z]{3,}", normalize_text(b)))

    if not b_words:
        return 0.0

    return len(a_words & b_words) / len(b_words)

### 6e — Reward #1: correctness

The heaviest grader. Think of it as a teacher marking against an answer key, out of 5.

It blends both scorers 50/50 — `2.5 * similarity + 2.5 * overlap` — so an answer needs *both* the
right content words and roughly the right phrasing to max out. Using only one is easy to game:
pure overlap rewards word-salad containing the right nouns; pure similarity rewards near-verbatim copying.

> **Production caveat:** matching a reference string is a *proxy* for correctness, not correctness.
> It teaches imitation. Real GRPO setups reward things that can be **checked** — unit tests passing,
> a math answer equalling the gold value, a JSON schema validating. Part 2 does exactly that.

In [ ]:
def correctness_reward(prompts, completions, reference_answer=None, **kwargs):
    """Reward #1 — score each completion against its reference answer. Range 0.0-5.0.

    Combines order-sensitive similarity and order-insensitive keyword recall in
    equal parts, so a high score requires both the right content and roughly the
    right phrasing.

    NOTE: this is a *proxy* reward suitable for a teaching demo. It rewards
    imitation of a reference string, not correctness. Production GRPO should
    reward something verifiable (see Part 2's `correctness_reward_func`).

    Args:
        prompts: batch of prompts (unused; required by the TRL signature).
        completions: the generated answers being graded.
        reference_answer: the dataset column, auto-passed by TRL as a list
            aligned with `completions`. Falls back to all-zero rewards if absent.
        **kwargs: other columns TRL forwards; must be accepted or TRL raises.

    Returns:
        list[float]: one reward per completion.
    """
    rewards = []

    if reference_answer is None:
        return [0.0 for _ in completions]

    for completion, ref in zip(completions, reference_answer):
        text = get_completion_text(completion)

        sim = similarity_score(text, ref)
        overlap = keyword_overlap_score(text, ref)

        # Combined correctness reward: 0 to 5
        score = (2.5 * sim) + (2.5 * overlap)
        rewards.append(float(score))

    return rewards

### 6f — Reward #2: helpfulness

A shape-and-effort grader that never looks at the reference. It encodes three heuristics:

| Rule | Δ | Intent |
|---|---|---|
| length in 40–260 chars | +1.0 | a real answer, not a fragment or an essay |
| length < 20 | −1.0 | one-word dodges |
| length > 350 | −0.5 | rambling |
| contains `.` or `,` | +0.5 | sentence-like, not a token dump |
| contains a hedge phrase | −1.5 | "I don't know" / "not sure" / "maybe" |

The hedge penalty is the interesting one — without it, small models discover that non-answers are
safe and converge on evasion.

In [ ]:
def helpfulness_reward(prompts, completions, **kwargs):
    """Reward #2 — score answers on completeness and non-evasiveness. Range -2.5 to +1.5.

    Reference-free. Pays for a sensible length and sentence-like punctuation;
    penalizes fragments, rambling, and hedge phrases. The hedge penalty matters:
    without it small models learn that "I'm not sure" is a low-risk answer and
    converge on evasion.

    Args:
        prompts: batch of prompts (unused; required by the TRL signature).
        completions: the generated answers being graded.
        **kwargs: other columns TRL forwards.

    Returns:
        list[float]: one reward per completion.
    """
    rewards = []

    bad_phrases = [
        "i don't know",
        "no idea",
        "maybe",
        "not sure",
        "random magic",
    ]

    for completion in completions:
        text = get_completion_text(completion)
        text_norm = normalize_text(text)

        score = 0.0

        # Good answer length
        if 40 <= len(text) <= 260:
            score += 1.0
        elif len(text) < 20:
            score -= 1.0
        elif len(text) > 350:
            score -= 0.5

        # Sentence-like response
        if "." in text or "," in text:
            score += 0.5

        # Penalize vague responses
        if any(bp in text_norm for bp in bad_phrases):
            score -= 1.5

        rewards.append(float(score))

    return rewards

### 6g — Reward #3: clarity

The smallest grader, and mostly a **guardrail**. Its job is to catch degenerate output — the
`"aaaaaaaaaa"` and `"!!!!!!!!!"` failure modes that RL loves to find when a reward has a loophole.

The regex `(.)\1{8,}` matches any character repeated 9+ times in a row.

Its length window (30–280) is deliberately narrower than `helpfulness_reward`'s (40–260 for the
bonus). Overlapping-but-not-identical windows are fine — summing several soft signals produces a
smoother reward landscape than one hard rule.

In [ ]:
def clarity_reward(prompts, completions, **kwargs):
    """Reward #3 — readability guardrail. Range -1.5 to +1.0.

    Mostly defensive: catches the degenerate outputs RL discovers when another
    reward has a loophole (character spam, empty/near-empty strings, runaway
    length). Small magnitude by design — it should shape, not dominate.

    Args:
        prompts: batch of prompts (unused; required by the TRL signature).
        completions: the generated answers being graded.
        **kwargs: other columns TRL forwards.

    Returns:
        list[float]: one reward per completion.
    """
    rewards = []

    for completion in completions:
        text = get_completion_text(completion)
        text_norm = normalize_text(text)

        score = 0.0

        # Has alphabetic content
        if re.search(r"[A-Za-z]", text):
            score += 0.5

        # Not too short / not too long
        if 30 <= len(text) <= 280:
            score += 0.5
        else:
            score -= 0.5

        # Penalize repeated junk: any character repeated 9+ times in a row.
        if re.search(r"(.)\1{8,}", text_norm):
            score -= 1.0

        rewards.append(float(score))

    return rewards

---
## Lesson 7 — `GRPOConfig`

The settings that actually matter for GRPO, as opposed to generic `TrainingArguments`:

| Parameter | Value | What it controls |
|---|---|---|
| **`num_generations`** | 4 | **The `G` in "group".** Completions sampled per prompt. This *is* the algorithm — with `G=1` there's no group to compare against and the advantage is undefined. Higher = lower-variance advantages, linearly more compute. 4–8 typical, 8–16 for hard tasks. |
| **`beta`** | 0.01 | KL penalty weight against the frozen reference. Too low → the model drifts and forgets language; too high → it can't move. Recent TRL defaults to `0.0`; a small non-zero value is safer while learning. |
| **`temperature` / `top_p`** | 0.9 / 0.95 | Rollout sampling. **Higher than you'd use at inference** — GRPO needs *diverse* completions. Low temperature makes all `G` samples near-identical, the group std collapses, and advantages go to zero. |
| **`max_completion_length`** | 80 | Rollout token cap. The main VRAM knob. |
| **`learning_rate`** | 1e-5 | RL is unstable; 1e-6 to 1e-5 is the usual band. |

**The batch-size constraint:** `per_device_train_batch_size` must be divisible by `num_generations`.
The trainer builds batches out of whole groups, and a partial group has no valid baseline. Here both
are 4 → one prompt per step, four completions of it.

`max_steps=10` is a smoke test. Expect nothing but a working loop.

In [22]:
training_args = GRPOConfig(
    output_dir="grpo_output",

    learning_rate=1e-5,

    # Important:
    # In TRL GRPO, per_device_train_batch_size should be divisible by num_generations.
    # So we keep both as 4 for simple single-GPU Colab demo.
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    num_generations=4,

    max_completion_length=80,

    temperature=0.9,
    top_p=0.95,

    # KL penalty. 0.0 is default in recent TRL GRPO, but 0.01 is okay for teaching demo.
    beta=0.01,

    max_steps=10,

    logging_steps=1,
    save_steps=10,

    remove_unused_columns=False,
    report_to="none",

    gradient_checkpointing=False,

    fp16=torch.cuda.is_available(),
)

## Lesson 8 — Build the trainer and train

Two things worth naming:

- **`reward_funcs`** takes a *list*. TRL calls each one on every batch and sums the results. It also
  logs each function separately as `rewards/<function_name>` — which is how you diagnose reward
  hacking. If `helpfulness_reward` climbs while `correctness_reward` flatlines, the model found the
  length bonus and stopped trying to be right.
- **`peft_config`** is passed to the *trainer*, not applied to the model beforehand. TRL wraps the
  model itself, which lets it toggle adapters off to get the KL reference model for free.

### What one training step does

1. Take a prompt from the batch.
2. Sample `G=4` completions from the current policy.
3. Score all 4 with all 3 reward functions; sum per completion.
4. Normalize within the group → advantages.
5. Policy-gradient update on the LoRA weights, plus the KL penalty.

### Reading the logs

- **`reward`** — mean total reward. Should trend up.
- **`reward_std`** — spread within groups. If it hits ~0 the model has collapsed to one answer and
  learning stops. This is the number to watch.
- **`kl`** — divergence from the reference. Steadily climbing means drift; raise `beta`.

In [23]:
trainer = GRPOTrainer(
    model=MODEL,
    processing_class=tokenizer,
    reward_funcs=[
        correctness_reward,
        helpfulness_reward,
        clarity_reward,
    ],
    args=training_args,
    train_dataset=dataset,
    peft_config=lora_config,
)

### 8a — Version check (troubleshooting)

`GRPOConfig`'s signature changes between TRL releases — arguments get renamed and removed fairly
often. If the config cell above raised a `TypeError` about an unexpected keyword, run this and
compare against what you passed.

In [24]:
import trl
import inspect
from trl import GRPOConfig

print("TRL version:", trl.__version__)
print(inspect.signature(GRPOConfig.__init__))

TRL version: 1.9.2
(self, output_dir: str | None = None, per_device_train_batch_size: int = 8, num_train_epochs: float = 3.0, max_steps: int = -1, learning_rate: float = 1e-06, lr_scheduler_type: transformers.trainer_utils.SchedulerType | str = 'linear', lr_scheduler_kwargs: dict | str | None = None, warmup_steps: float = 0, optim: transformers.training_args.OptimizerNames | str = 'adamw_torch_fused', optim_args: str | None = None, weight_decay: float = 0.0, adam_beta1: float = 0.9, adam_beta2: float = 0.999, adam_epsilon: float = 1e-08, optim_target_modules: None | str | list[str] = None, gradient_accumulation_steps: int = 1, average_tokens_across_devices: bool = True, max_grad_norm: float = 1.0, label_smoothing_factor: float = 0.0, bf16: bool | None = None, fp16: bool = False, bf16_full_eval: bool = False, fp16_full_eval: bool = False, tf32: bool | None = None, gradient_checkpointing: bool = True, gradient_checkpointing_kwargs: dict[str, typing.Any] | str | None = None, torch_compile

In [25]:
trainer.train()

Step,Training Loss
1,-0.000000
2,0.000000
3,0.000000
4,0.276096
5,0.160381
6,-0.008093
7,0.028336
8,-0.000001
9,0.000000
10,0.153412


TrainOutput(global_step=10, training_loss=0.06101317333057921, metrics={'train_runtime': 65.7567, 'train_samples_per_second': 0.608, 'train_steps_per_second': 0.152, 'total_flos': 0.0, 'train_loss': 0.06101317333057921, 'epoch': 1.25})

### 8b — Save the adapter

`save_model` writes **only the LoRA adapter** (a few MB), not the full model. Loading it back
requires the base model plus this directory — see the next cell. The tokenizer is saved alongside
so the adapter folder is self-describing.

In [26]:
trainer.save_model("grpo_output_final")
tokenizer.save_pretrained("grpo_output_final")

('grpo_output_final/tokenizer_config.json',
 'grpo_output_final/chat_template.jinja',
 'grpo_output_final/tokenizer.json')

## Lesson 9 — Inference

Reload the base model and stack the trained adapter on top with `PeftModel.from_pretrained`.
`model.eval()` disables dropout.

**The prompt template must match training exactly.** GRPO conditioned the learned behaviour on
`"Question: ...\nAnswer in simple English:"`. Change it and you're querying an off-distribution
model that will look untrained.

Sampling is more conservative here (`temperature=0.7`, `top_p=0.9`) than during rollouts — training
wanted diversity, inference wants the model's best guess.

In [27]:
base_model_id = "HuggingFaceTB/SmolLM-135M-Instruct"
adapter_path = "grpo_output_final"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map="auto",
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(49152, 576, padding_idx=2)
        (layers): ModuleList(
          (0-29): 30 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=576, out_features=576, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=576, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=576, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): Line

In [ ]:
def generate_answer(question, max_new_tokens=100):
    """Generate an answer with the GRPO-trained adapter.

    The prompt template here is IDENTICAL to `format_prompt` in Lesson 3 — the
    trained behaviour is conditioned on that exact prefix, so any change makes
    the model look untrained.

    Args:
        question: raw question text, without any template.
        max_new_tokens: generation cap. Training used max_completion_length=80,
            so much longer outputs are extrapolation.

    Returns:
        str: the decoded generation, INCLUDING the prompt prefix (special tokens
        stripped). Slice it off if you want only the answer.
    """
    prompt = f"Question: {question}\nAnswer in simple English:"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,   # lower than the 0.9 used for rollouts
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [32]:
print(generate_answer("Explain AI in simple terms."))

Question: Explain AI in simple terms.
Answer in simple English: AI is a type of computer program that can perform tasks that typically require human intelligence, such as:

**Examples:**

1. **Image recognition**: AI-powered systems can recognize objects, people, and scenes in images, such as recognizing a dog in a picture.
2. **Natural Language Processing (NLP)**: AI-powered systems can understand and generate human-like language, such as translating text from one language to another.
3. **Robotics**: AI-


> **Expect underwhelming results.** 135M parameters, 8 examples, 10 steps. Part 1 proves the
> pipeline runs; it does not produce a good model. Part 2 is where the training is real.

---
---
# Part 2 — GRPO on GSM8K (Unsloth + vLLM)

This is the DeepSeek-R1-style setup, scaled to a T4.

**What changes from Part 1, and why it matters:**

| | Part 1 | Part 2 |
|---|---|---|
| Model | SmolLM-135M | Qwen2.5-0.5B-Instruct (4-bit) |
| Rollouts | HF `generate` | **vLLM** — 5–10× faster |
| Reward | similarity to a reference | **verifiable**: does the final number equal the gold answer? |
| Output shape | free text | enforced `<reasoning>` / `<answer>` XML |
| Steps | 10 | 100 |

The verifiable reward is the substantive upgrade. `2.0` if the extracted number matches, `0.0`
otherwise — nothing to game, no proxy, no imitation. That's the property that made R1's training work.

**Rollout speed is the bottleneck in GRPO.** With `G=4` completions per prompt over 100 steps that's
400 generations before you count anything else. vLLM's paged attention and continuous batching are
what make this feasible on a free GPU.

> **Restart the runtime after the install cell.** Unsloth patches Transformers at import time and
> will not take effect on an already-imported module.

## Lesson 10 — Install Unsloth + vLLM

In [33]:
!pip install -U unsloth vllm
!pip install -U transformers datasets trl accelerate peft bitsandbytes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 4.1 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of vllm to determine which version is compatible with other requirements. This could take a while.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.6 MB/s eta 0:00:00
INFO: pip is looking at multiple versions of quack-kernels to determine which version is compatible with other requirements. This could take a while.
INFO: pip is looking at multiple versions of cuda-tile[tileiras] to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.1/274.1 MB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 90.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.7/211.7 kB 24.1 MB/s eta 0:00

  Using cached transformers-5.14.1-py3-none-any.whl.metadata (32 kB)
  Using cached datasets-5.0.1-py3-none-any.whl.metadata (23 kB)
  Using cached trl-1.9.2-py3-none-any.whl.metadata (12 kB)
Using cached transformers-5.14.1-py3-none-any.whl (11.6 MB)
Using cached datasets-5.0.1-py3-none-any.whl (559 kB)
Using cached trl-1.9.2-py3-none-any.whl (889 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 780.4/780.4 kB 20.1 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
  Attempting uninstall: datasets
    Found existing installation: datasets 4.3.0
    Uninstalling datasets-4.3.0:
      Successfully uninstalled datasets-4.3.0
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.6
    Uninstalling transformers-4.57.6:
      Successfully uninstalled transformers-4.57.6
ERROR: Operation cancelled by user

## Lesson 11 — Load the model with fast inference enabled

`FastLanguageModel.from_pretrained` differs from the plain Transformers loader in ways that matter here:

- **`load_in_4bit=True`** — QLoRA. Base weights in 4-bit NF4, adapters in bf16. Roughly 4× less VRAM.
- **`fast_inference=True`** — routes rollout generation through **vLLM**. This is the flag that makes
  GRPO practical on a T4.
- **`max_lora_rank`** — must be declared up front so vLLM can pre-allocate for the adapter.
- **`gpu_memory_utilization=0.6`** — caps vLLM's KV-cache pool at 60% of VRAM, leaving room for
  training gradients and optimizer state. **Lower this to 0.5 if you hit OOM**; raise toward 0.7 for
  faster rollouts if you have headroom. It's the single most useful knob when things won't fit.

In [2]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 1024
lora_rank = 16

MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=max_seq_length,
    load_in_4bit=True,
    fast_inference=True,
    max_lora_rank=lora_rank,
    gpu_memory_utilization=0.6,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Exception: cannot import name 'dependency_versions_check' from partially initialized module 'transformers' (most likely due to a circular import) (/usr/local/lib/python3.12/dist-packages/transformers/__init__.py)

## Lesson 12 — Attach LoRA adapters

Broader coverage than Part 1: all four attention projections **plus** the three MLP projections.
Reasoning behaviour lives substantially in the MLPs, so `q_proj`/`v_proj` alone under-fits here.

`use_gradient_checkpointing="unsloth"` is Unsloth's own implementation — it trades a little compute
for a large activation-memory saving, and is what lets 0.5B + 4 rollouts fit alongside a KV cache.

`random_state=3407` makes adapter init reproducible.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=lora_rank,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=lora_rank,
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)

## Lesson 13 — The XML reasoning format

We force a structured output:

```
<reasoning>
Step-by-step working goes here.
</reasoning>
<answer>
42
</answer>
```

**Why bother?** Two reasons.

1. **It makes the answer extractable.** A verifiable reward needs to isolate the final number
   unambiguously. Regexing a number out of free-flowing prose is unreliable — is it the answer, or
   an intermediate value?
2. **It creates room to think.** The `<reasoning>` block gives the model tokens to compute in before
   committing. This is the mechanism behind chain-of-thought, and GRPO's format rewards are how you
   *train* the model to use it rather than just prompting for it.

> ### ⚠️ Fix applied here
> In the original notebook the XML tags were **missing from every string and regex in Part 2** —
> `SYSTEM_PROMPT`, `XML_COT_FORMAT`, `extract_xml_answer`, both format reward patterns, and
> `count_xml` all had empty tags where `<reasoning>` / `<answer>` should have been. That's the
> classic symptom of pasting code out of a rendered HTML page, which eats anything in angle brackets.
> Left as-is, `extract_xml_answer` splits on the empty string, the format rewards match everything,
> and the correctness reward can never fire. **The tags have been restored throughout Part 2.**

In [ ]:
SYSTEM_PROMPT = """
Respond in the following format:

<reasoning>
Write your step-by-step reasoning here.
</reasoning>
<answer>
Write only the final numeric answer here.
</answer>
"""

# Template used to build few-shot examples or to check what a well-formed
# response looks like. The strict format reward below matches this exactly.
XML_COT_FORMAT = """\
<reasoning>
{reasoning}
</reasoning>
<answer>
{answer}
</answer>
"""

## Lesson 14 — GSM8K

**GSM8K** is 8.5K grade-school math word problems. Its gold answers use a fixed convention: the
full worked solution, then `####`, then the final number.

```
Natalia sold 48 clips in April, and half as many in May...
48 / 2 = 24
48 + 24 = 72
#### 72
```

We split on `####` to get just `72` — that becomes the ground truth the correctness reward checks
against. **This is why GSM8K is the standard GRPO benchmark**: the answer is a bare number, so
correctness is a string comparison, not a judgement call.

The prompts are built in **chat format** (a list of role/content dicts) rather than plain strings,
which is why Part 2's reward functions index `completion[0]["content"]` where Part 1 used
`get_completion_text`.

In [ ]:
import re
from datasets import load_dataset, Dataset


def extract_hash_answer(text: str):
    """Pull the final answer out of a GSM8K gold solution.

    GSM8K solutions end with "#### <number>". Everything before it is the worked
    reasoning, which we discard — only the final value is used for grading.

    Args:
        text: the raw "answer" field from GSM8K.

    Returns:
        str | None: the final answer with whitespace stripped, or None if the
        row is malformed (no "####" marker).
    """
    if "####" not in text:
        return None
    return text.split("####")[-1].strip()


def get_gsm8k_questions(split="train") -> Dataset:
    """Load GSM8K and reshape it into the columns GRPOTrainer expects.

    Produces two columns:
      * "prompt" — chat-format messages: the system prompt (which specifies the
        XML output format) plus the user's question.
      * "answer" — the bare gold number, forwarded by TRL to any reward function
        that declares an `answer` parameter.

    Args:
        split: "train" or "test".

    Returns:
        Dataset: ready to pass as `train_dataset`.
    """
    data = load_dataset("openai/gsm8k", "main")[split]

    data = data.map(
        lambda x: {
            "prompt": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": x["question"]},
            ],
            "answer": extract_hash_answer(x["answer"]),
        }
    )

    return data


dataset = get_gsm8k_questions("train")

print(dataset[0])

---
## Lesson 15 — The reward stack

Five reward functions, summed. Notice the **ladder of difficulty** — this is the important design
idea in the whole notebook:

| Function | Max | What it grades |
|---|---|---|
| `xmlcount_reward_func` | ~0.5 | partial credit per correct tag |
| `soft_format_reward_func` | 0.5 | tags present in the right order |
| `strict_format_reward_func` | 0.5 | exact whitespace-perfect layout |
| `int_reward_func` | 0.5 | the answer field is a number at all |
| `correctness_reward_func` | **2.0** | the number is **right** |

**Why five instead of one?** Because a lone correctness reward is a *sparse* signal. Early on, a
0.5B model gets nearly every problem wrong, every completion in the group scores 0.0, the group std
is 0, advantages are 0, and **no gradient flows**. Training never starts.

The format rewards are dense — they fire on almost every rollout and provide a gradient from step 1.
The model first learns to emit well-formed XML, *then* to put a number in it, *then* to get the
number right. It's curriculum learning encoded in the reward function. `xmlcount_reward_func` gives
fractional credit for each individual tag, so even the very first malformed attempts get a signal.

Correctness is weighted 4× everything else, so once the model can produce format reliably, the only
way left to improve is to actually solve the problem.

### 15a — Extract the answer

In [ ]:
def extract_xml_answer(text: str) -> str:
    """Pull the contents of the <answer> block out of a model completion.

    Takes the text after the LAST <answer> (so stray tags in the reasoning don't
    break it), then everything before the following </answer>.

    Args:
        text: the model's full completion.

    Returns:
        str: the stripped answer text, or "" if no <answer> tag is present —
        which scores 0.0 downstream rather than raising.
    """
    if "<answer>" not in text:
        return ""
    answer = text.split("<answer>")[-1]
    answer = answer.split("</answer>")[0]
    return answer.strip()

### 15b — Reward: correctness (verifiable, weight 2.0)

**The reward that matters.** Exact string match between the extracted answer and the GSM8K gold
value: `2.0` or `0.0`, nothing in between.

This is the crucial difference from Part 1. There is no proxy, no similarity metric, no imitation —
the answer is either right or it isn't. A model cannot reward-hack arithmetic. Everything else in
this stack exists to get the model to a state where *this* reward can start firing.

`answer` arrives automatically as a keyword argument because the dataset has a column of that name.

> ### ⚠️ Fix applied here
> The original function built the `rewards` list and then **never returned it**, so it returned
> `None` and `trainer.train()` would crash on the first step. `return rewards` has been added.
> A `print` of one sample per batch is also included — reading generations during training is how
> you catch reward hacking, and it's cheap insurance.

In [ ]:
def correctness_reward_func(prompts, completions, answer, **kwargs) -> list[float]:
    """Reward: exact-match correctness against the GSM8K gold answer. 2.0 or 0.0.

    The only VERIFIABLE reward in the stack, and the reason this setup works
    where Part 1's similarity proxy doesn't — arithmetic can't be gamed. Weighted
    4x the format rewards so it dominates once the model can produce valid XML.

    Args:
        prompts: batch of chat-format prompts (used only for the debug print).
        completions: chat-format completions from the policy.
        answer: gold answers, auto-forwarded by TRL from the dataset column.
        **kwargs: other columns TRL forwards.

    Returns:
        list[float]: 2.0 where the extracted answer matches gold, else 0.0.
    """
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]

    # Peek at one rollout per batch. Reading real generations is how you notice
    # reward hacking and format drift before the curves tell you.
    print(
        "-" * 20,
        f"\nQuestion:\n{prompts[0][-1]['content']}",
        f"\nGold answer:\n{answer[0]}",
        f"\nModel response:\n{responses[0]}",
        f"\nExtracted:\n{extracted_responses[0]}",
    )

    rewards = []
    for model_answer, gold_answer in zip(extracted_responses, answer):
        if model_answer == gold_answer:
            rewards.append(2.0)
        else:
            rewards.append(0.0)

    return rewards

### 15c — Reward: integer check (weight 0.5)

A stepping stone between "produced XML" and "produced the right number". It pays for the answer
field containing a bare digit string, regardless of value.

Without it there's a hard cliff — the model can nail the format perfectly and still get zero signal
until it happens to solve a problem correctly. This makes the intermediate step *"put a number
here"* rewarding on its own.

`.isdigit()` is strict: it rejects `"42.5"`, `"-3"`, and `"1,000"`. Fine for GSM8K, whose answers are
non-negative integers — but you'd need to loosen it for other datasets.

In [ ]:
def int_reward_func(completions, **kwargs) -> list[float]:
    """Reward: the <answer> block contains a bare integer. 0.5 or 0.0.

    A stepping stone between "valid XML" and "correct answer" — it removes the
    hard cliff where a well-formatted but wrong response earns nothing.

    Note `.isdigit()` is strict: "42.5", "-3" and "1,000" all fail. That suits
    GSM8K (non-negative integer answers) but needs loosening elsewhere.

    Args:
        completions: chat-format completions from the policy.
        **kwargs: other columns TRL forwards.

    Returns:
        list[float]: 0.5 per numeric answer, else 0.0.
    """
    responses = [completion[0]["content"] for completion in completions]
    extracted_responses = [extract_xml_answer(r) for r in responses]

    return [0.5 if r.isdigit() else 0.0 for r in extracted_responses]

### 15d — Reward: soft format (weight 0.5)

Checks that all four tags appear **in the right order**, tolerating any whitespace between blocks.
`re.DOTALL` makes `.` match newlines so the pattern spans multi-line reasoning.

This is the forgiving format check — it fires as soon as the model has the *structure* right, even
if the layout is scruffy. `strict_format_reward_func` then pays extra for getting the layout exact.

> ### ⚠️ Fix applied here
> Two problems: the regex had no tags (`r".*?\s*.*?"` matches literally anything, so this returned
> 0.5 unconditionally), and the final list comprehension was **missing its closing bracket** — a
> `SyntaxError` that would stop the notebook dead. Both corrected.

In [ ]:
def soft_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward: the four XML tags appear in the correct order. 0.5 or 0.0.

    Whitespace-tolerant — this is the forgiving structural check. Uses re.search
    (match anywhere) and re.DOTALL so the pattern spans multi-line reasoning.

    Args:
        completions: chat-format completions from the policy.
        **kwargs: other columns TRL forwards.

    Returns:
        list[float]: 0.5 per well-ordered response, else 0.0.
    """
    pattern = r"<reasoning>.*?</reasoning>\s*<answer>.*?</answer>"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.search(pattern, r, re.DOTALL) for r in responses]

    return [0.5 if match else 0.0 for match in matches]

### 15e — Reward: strict format (weight 0.5)

The exacting version: `re.match` anchors at the start, `^`/`$` anchor both ends, and every newline
must be exactly where `XML_COT_FORMAT` puts it. No leading preamble, no trailing commentary.

Soft and strict stack — a perfectly formatted response earns both (1.0 total), a scruffy one earns
only the soft reward (0.5). That gradient is what pulls the model from "roughly right" to "exactly right".

> ### ⚠️ Fix applied here
> The pattern was `r"^\n.*?\n\n\n.*?\n\n$"` — tags missing, so it matched nothing usable. Restored.

In [ ]:
def strict_format_reward_func(completions, **kwargs) -> list[float]:
    """Reward: the response matches XML_COT_FORMAT exactly. 0.5 or 0.0.

    Anchored at both ends with re.match plus ^/$ — no preamble, no trailing
    commentary, newlines exactly where the template puts them. Stacks with
    `soft_format_reward_func`, so a perfect response earns 1.0 across the two.

    Args:
        completions: chat-format completions from the policy.
        **kwargs: other columns TRL forwards.

    Returns:
        list[float]: 0.5 per exactly-formatted response, else 0.0.
    """
    pattern = r"^<reasoning>\n.*?\n</reasoning>\n<answer>\n.*?\n</answer>\n$"
    responses = [completion[0]["content"] for completion in completions]
    matches = [re.match(pattern, r, re.DOTALL) for r in responses]

    return [0.5 if match else 0.0 for match in matches]

### 15f — Reward: XML tag counting (weight ~0.5, fractional)

The **densest** signal in the stack, and the one that gets training off the ground.

Rather than all-or-nothing, it awards `0.125` per correctly-placed tag — so a response with just
`<reasoning>` and `</reasoning>` still earns 0.25. On step 1, when nothing matches the strict
patterns, this is often the *only* non-zero reward, and it's what keeps the group std above zero so
gradients can flow.

It also applies a small trailing-text penalty (`* 0.001` per character) after the closing tags,
discouraging the model from appending commentary after `</answer>`.

> ### ⚠️ Fix applied here
> Same two issues as `soft_format_reward_func`: all tags missing from the `.count()` and `.split()`
> calls, and the final comprehension **missing its closing bracket**. Both corrected.

In [ ]:
def count_xml(text) -> float:
    """Award fractional credit (0.125) for each correctly-placed XML tag.

    The densest reward in the stack. Where the format rewards are all-or-nothing,
    this pays partial credit, so even a half-formed early rollout gets signal —
    which is what keeps the group std above zero and lets gradients flow at step 1.

    Also subtracts a small per-character penalty for text trailing after the
    closing tags, discouraging commentary appended past </answer>.

    Args:
        text: a single completion string.

    Returns:
        float: roughly 0.0-0.5, minus any trailing-text penalty.
    """
    count = 0.0

    if text.count("<reasoning>\n") == 1:
        count += 0.125

    if text.count("\n</reasoning>\n") == 1:
        count += 0.125

    if text.count("\n<answer>\n") == 1:
        count += 0.125
        # Penalize anything written after the answer block opens elsewhere.
        count -= len(text.split("\n</answer>\n")[-1]) * 0.001

    if text.count("\n</answer>") == 1:
        count += 0.125
        count -= (len(text.split("\n</answer>")[-1]) - 1) * 0.001

    return count


def xmlcount_reward_func(completions, **kwargs) -> list[float]:
    """Reward: fractional per-tag credit. See `count_xml`.

    Args:
        completions: chat-format completions from the policy.
        **kwargs: other columns TRL forwards.

    Returns:
        list[float]: one partial-credit score per completion.
    """
    contents = [completion[0]["content"] for completion in completions]
    return [count_xml(c) for c in contents]

---
## Lesson 16 — Training configuration

Differences from Part 1's config, and why:

| Setting | Value | Reasoning |
|---|---|---|
| `learning_rate=5e-6` | half of Part 1's | Bigger model, longer run — RL destabilizes easily. |
| `optim="paged_adamw_8bit"` | 8-bit paged Adam | Optimizer state in 8-bit, paged to CPU on pressure. Big VRAM saving. |
| `max_grad_norm=0.1` | very aggressive clipping | **Essential for GRPO.** A single high-advantage rollout can produce a huge gradient and wreck the policy. Tight clipping is the standard stability fix. |
| `adam_beta2=0.99` | down from 0.999 | Faster adaptation to the noisy, non-stationary reward signal. |
| `warmup_ratio=0.1` + cosine | LR schedule | Warmup avoids early instability while rollouts are near-random. |
| `weight_decay=0.1` | regularization | Standard for LoRA runs at this scale. |
| `max_prompt_length=256` | prompt cap | Prompt + completion must fit `max_seq_length` (1024), so completions get the remaining 768. |
| `max_steps=100` | demo length | **Use 250+ for results you'd actually believe.** Format rewards usually saturate around step 50–100; correctness starts moving after that. |

`per_device_train_batch_size=1` with `num_generations=4`: one prompt per step, four completions of it.

In [ ]:
from trl import GRPOConfig, GRPOTrainer

max_prompt_length = 256

training_args = GRPOConfig(
    learning_rate=5e-6,
    adam_beta1=0.9,
    adam_beta2=0.99,
    weight_decay=0.1,
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    optim="paged_adamw_8bit",

    logging_steps=1,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=1,

    num_generations=4,  # increase to 6 if GPU memory allows

    max_prompt_length=max_prompt_length,
    max_completion_length=max_seq_length - max_prompt_length,

    max_steps=100,       # demo; use 250+ for better result
    save_steps=100,

    max_grad_norm=0.1,
    report_to="none",
    output_dir="grpo_outputs",
)

## Lesson 17 — Train

**What to watch in the logs** — TRL logs each reward function separately, and the order in which
they rise tells the story:

1. **First** `rewards/xmlcount_reward_func` climbs — the model is learning tags.
2. **Then** `rewards/soft_format_reward_func` and `strict_format_reward_func` — layout is locking in.
3. **Then** `rewards/int_reward_func` — it's putting numbers in the answer block.
4. **Last, and slowest,** `rewards/correctness_reward_func` — it's actually solving problems.

If correctness stays at 0.0 while formats are saturated, the model has learned to *look* right
without reasoning. Options: train longer, raise the correctness weight, or move to a larger base model.

Also watch **`reward_std`**. If it collapses toward zero, all four completions in each group have
become identical, advantages vanish, and learning stops. The fix is usually more rollout diversity
(higher temperature) or more generations per group.

100 steps on a T4 takes roughly 20–40 minutes.

In [ ]:
trainer = GRPOTrainer(
    model=model,
    processing_class=tokenizer,
    reward_funcs=[
        xmlcount_reward_func,
        soft_format_reward_func,
        strict_format_reward_func,
        int_reward_func,
        correctness_reward_func,
    ],
    args=training_args,
    train_dataset=dataset,
)

In [ ]:
trainer.train()

## Lesson 18 — Save and test

`save_lora` writes just the adapter — a few MB. The base model is untouched.

In [ ]:
model.save_lora("grpo_saved_lora")

### 18a — Fast inference with vLLM

`fast_generate` runs generation through vLLM rather than HF's `generate`. `load_lora` hot-loads the
adapter into the running engine — no reload, and you can swap between adapters (or compare against
the base model by omitting `lora_request`) at will.

**`apply_chat_template` is required.** The model was trained on chat-formatted prompts with the
system message attached; a raw string won't reproduce the trained behaviour.

A correct response should come back wrapped in `<reasoning>` and `<answer>` tags.

In [ ]:
from vllm import SamplingParams

test_prompt = tokenizer.apply_chat_template(
    [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": "If there are 3 boxes and each box has 4 apples, how many apples are there in total?"},
    ],
    tokenize=False,
    add_generation_prompt=True,
)

sampling_params = SamplingParams(
    temperature=0.7,
    top_p=0.95,
    max_tokens=512,
)

output = model.fast_generate(
    test_prompt,
    sampling_params=sampling_params,
    lora_request=model.load_lora("grpo_saved_lora"),
)[0].outputs[0].text

print(output)

### 18b — Export a merged model

Folds the LoRA weights into the base model and saves a standalone 16-bit checkpoint — loadable with
plain `AutoModelForCausalLM`, no PEFT required, and servable by vLLM/TGI directly.

**Trade-off:** the merged model is full-size (~1GB for 0.5B in fp16) versus a few MB for the adapter,
and you lose the ability to swap adapters at runtime. Merge for deployment; keep adapters for
iteration.

Other `save_method` options: `"merged_4bit"` (smaller, some quality loss) and `"lora"` (adapter only).

In [ ]:
model.save_pretrained_merged(
    "grpo_merged_model",
    tokenizer,
    save_method="merged_16bit"
)

---
## Where to go next

**Make the run real**
- `max_steps` → 250–500. 100 is a smoke test.
- `num_generations` → 8. Lower-variance advantages, better learning; linearly more compute.
- Swap in Qwen2.5-1.5B or 3B-Instruct if you have an A100.
- Evaluate on the GSM8K **test** split — training reward is not accuracy.

**Learn the algorithm properly**
- Set `num_generations=2` and watch training destabilize. That's the group baseline doing its job.
- Set `beta=0.0` (no KL) and watch how fast the policy drifts off-distribution.
- Delete `xmlcount_reward_func` and see whether training still gets started.
- Log per-function rewards to Weights & Biases (`report_to="wandb"`) and read the curves.

**Write your own rewards**
The transferable skill here is reward design. The pattern that works: **one verifiable signal**
(tests pass, schema validates, answer matches) plus **dense format shaping** to bootstrap. Good
targets — code that must pass unit tests, tool calls that must match a JSON schema, SQL that must
return the right rows, extraction that must hit an exact field.

**Read**
- DeepSeekMath (arXiv 2402.03300) — the paper that introduced GRPO.
- DeepSeek-R1 (arXiv 2501.12948) — GRPO at scale, with emergent reasoning.
- TRL's `GRPOTrainer` docs — the config surface changes between releases.